# atommovr controller

```bash
uv sync --extra hardware
```

The `hardware` extra installs `spcm` (and on Linux, CUDA). See [setup_guide.md](setup_guide.md).


## 1. Settings

Allied Vision Alvium 1800 U-052: 816×624, 8-bit mono. Both cameras share `Camera.detect_occupancy` (blob → rotate → `fit_grid_and_assign`).


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from atommovr.utils.core import Configurations, PhysicalParams
from atommovr.utils.errormodels import UniformVacuumTweezerError
from atommovr_controller import (
    AtommovrController,
    GaussianCameraConfig,
    HardwareConfig,
    OfflineArrayCamera,
    RealArrayCamera,
    SoftwareConfig,
    configure_logging,
)
from awg_controller import AODSettings, AWGEngine, AWGEngineConfig, CardConfig
from recorder import Recorder

configure_logging()

In [ ]:
ROWS, COLS = 30, 30

# Offline physics only (ignored when engine is attached).
lab_error = UniformVacuumTweezerError(
    pickup_time=0.1e-6,
    putdown_time=0.1e-6,
    accel_time=0.1e-6,
    decel_time=0.1e-6,
    pickup_fail_rate=0.01,
    putdown_fail_rate=0.01,
    lifetime=5e3,
    seed=0,
)

lab_params = PhysicalParams(
    spacing=8e-6,  # overwritten from FOV below
    AOD_speed=3,  # µm/µs
    loading_prob=0.65,
    middle_size=[20, 20],  # must fit inside ROWS x COLS
)

aod = AODSettings(
    grid_rows=ROWS,
    grid_cols=COLS,
    f_min_v=85e6,
    f_max_v=121e6,
    f_min_h=85.5e6,
    f_max_h=121.5e6,
    alignment="center",
)

# dx/df = f_obj * (f1/f2) * (λ/v)
wavelength_m, v_acoustic = 808e-9, 650.0
f1_mm, f2_mm, f_obj_mm = 75.0, 400.0, 28.0
aod.um_per_mhz = (
    (f_obj_mm * 1e-3) * (f1_mm / f2_mm) * (wavelength_m / v_acoustic) * 1e12
)
lab_params.spacing = aod.fov_um_v / (aod.grid_cols - 1) * 1e-6

lab_camera = GaussianCameraConfig(
    image_shape=(624, 816),
    sigma_px=2,
    peak_counts=180.0,
    background=12.0,
    noise_level=5,
    stripe_intensity=1,
    min_spacing_px=15,
    spacing_x=20,
    spacing_y=20,
    angle=1,
    dtype=np.uint8,
)

ALGORITHM_NAME = "Hungarian"
# "PCFA", "Hungarian", "Tetris", "BalanceAndCompact",
# "BCv2", "ParallelLBAP", "ParallelHungarian", "GeneralizedBalance"

TARGET_TYPE = Configurations.MIDDLE_FILL
# MIDDLE_FILL, ZEBRA_HORIZONTAL, ZEBRA_VERTICAL, CHECKERBOARD, Left_Sweep, RANDOM

sw = SoftwareConfig(
    algorithm_name=ALGORITHM_NAME,
    max_rounds=5,
    target_type=TARGET_TYPE,
    error_model=lab_error,
)
hw = HardwareConfig(
    card_path="/dev/spcm0",
    max_amplitude_v=1.0,  # start conservative; raise toward <=1.6 after a scope check
    output_load_ohms=50.0,
    aod_settings=aod,
    physical_params=lab_params,
)

print(f"lattice = {aod.grid_rows}x{aod.grid_cols}")
print(
    f"Δf_v = {aod.f_spacing_v / 1e6:.3f} MHz/site, "
    f"Δf_h = {aod.f_spacing_h / 1e6:.3f} MHz/site"
)
print(f"um_per_mhz = {aod.um_per_mhz:.3f}, spacing = {lab_params.spacing * 1e6:.2f} µm")
print(f"algorithm = {ALGORITHM_NAME!r}, target = {TARGET_TYPE.name}")

preview = OfflineArrayCamera(
    (ROWS, COLS),
    image_generator=lab_camera,
    physical_params=lab_params,
    seed=42,
)
frame = preview.acquire()
print(
    f"frame {frame.shape} {frame.dtype}, "
    f"truth {int(preview.occupancy.sum())}, "
    f"detected {int(preview.detect_occupancy(frame).sum())}"
)
fig, ax = plt.subplots(figsize=(6, 4))
ax.imshow(frame, cmap="gray", origin="upper")
ax.set_xlabel("x (px)")
ax.set_ylabel("y (px)")
plt.show()

## 2. Simulation

`engine=None` (the default): log + sleep, no card. `Recorder` writes `runs/run_*/meta.json` and one JSON line per round to `rounds.jsonl`.


In [ ]:
grid = (aod.grid_rows, aod.grid_cols)
recorder = Recorder(
    "runs",
    meta={
        "grid": list(grid),
        "target": list(lab_params.middle_size),
        "algo": ALGORITHM_NAME,
        "seed": 0,
        "note": "simulation",
    },
)
offline_cam = OfflineArrayCamera(
    grid,
    image_generator=lab_camera,
    physical_params=lab_params,
    seed=0,
)

with AtommovrController(sw, hw, camera=offline_cam, hooks=[recorder]) as ctrl:
    ok = ctrl.run()
    mask = (ctrl.array.target[:, :, 0] > 0).astype(int)
    occ = offline_cam.occupancy

print(f"success={ok}")
print(f"check {recorder.run_dir}")

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(mask, cmap="Blues", origin="upper", vmin=0, vmax=1)
axes[0].set_title(f"Target ({TARGET_TYPE.name})")
if occ is not None:
    axes[1].imshow(occ, cmap="Blues", origin="upper")
    filled = int((occ * mask).sum())
    axes[1].set_title(f"Final occ ({filled}/{int(mask.sum())} target sites)")
plt.tight_layout()
plt.show()

## 3. Hardware

```text
Alvium (or OfflineArrayCamera)
        │ acquire()
        ▼
Camera.detect_occupancy  →  occupancy grid
        │
        ▼
Algorithm  →  move batches
        │
        ▼
RFConverter  →  AWGBatch
        │
        ▼
AWGEngine  →  Spectrum AWG  →  AOD RF
```

Before connecting the AOD amp:

- `HardwareConfig.max_amplitude_v` must stay ≤ 2.0 V (default 1.6 V). Start at 1.0 V and check on a scope.
- `AWGEngine` needs `spcm` and `cupy`/CUDA; missing either raises when the engine cell is uncommented.
- Leave `engine = None` to stay in simulation. Uncomment `AWGEngine(...)` to drive the card.
- Each round is `stop` → `load_round` → `play` (one phase-continuous waveform; `stop` is a real RF gap).


In [ ]:
grid = (aod.grid_rows, aod.grid_cols)

# Uncomment when the Alvium is hooked up.
# def alvium_grab() -> np.ndarray:
#     raise NotImplementedError("hook up the Alvium SDK here")
#
# cam = RealArrayCamera(grid, camera_fn=alvium_grab)
cam = OfflineArrayCamera(
    grid,
    image_generator=lab_camera,
    physical_params=lab_params,
    seed=0,
)

engine = None
# engine = AWGEngine(
#     CardConfig(
#         card_path=hw.card_path,
#         max_amplitude_v=hw.max_amplitude_v,
#         output_load_ohms=hw.output_load_ohms,
#         aod_settings=hw.aod_settings,
#     ),
#     AWGEngineConfig(mode="memory"),  # experiment rounds are ms-long → full 1.25 GS/s
# )

hw_recorder = Recorder(
    "runs",
    meta={
        "grid": list(grid),
        "target": list(lab_params.middle_size),
        "algo": ALGORITHM_NAME,
        "note": "hardware" if engine is not None else "hardware (engine=None)",
    },
)

with AtommovrController(sw, hw, camera=cam, hooks=[hw_recorder], engine=engine) as ctrl:
    ok = ctrl.run()

print(f"success={ok}")
print(f"check {hw_recorder.run_dir}")